# Reto de Inteligencia Artificial y Machine Learning - Nivel Medio
## Análisis y Clasificación del Riesgo Sísmico Territorial en Santander con Modelos No Supervisados
**Institución:** Centro de Innovación e Inteligencia Artificial (CIAI) · Universitaria Tecnológica de Santander (UTS)
**Categoría:** Nivel Medio (Clasificación e Interpretación Territorial - Sala 315A)
**Modelo Obligatorio:** Únicamente Modelos No Supervisados
**Fuente de Datos Oficial:** Servicio Geológico Colombiano (SGC) / RSNC / USGS FDSN Web Service & DANE / IGAC
**Periodo de Estudio:** 2004 - 2026 (20+ años de comportamiento sísmico instrumental de Santander)
---


### 1. Justificación Geotectónica del Problema
Santander es el epicentro de la actividad sísmica en Colombia. Bajo la Mesa de Los Santos se localiza el **Nido Sísmico de Bucaramanga**, una de las fuentes de sismicidad intermedia más activas del planeta (~150 km de profundidad), generando cientos de eventos al mes.

**Objetivo del Proyecto:** Demostrar cómo modelos de aprendizaje no supervisado (K-Means, DBSCAN, GMM y Agrupamiento Jerárquico) logran descubrir patrones latentes de liberación de energía y profundidad, y cómo estos patrones se traducen en un **Índice Compuesto de Riesgo Sísmico Territorial (ICRST)** para los municipios de Santander.

In [ ]:
# Importacion de librerias cientificas y geoespaciales
import os, sys, json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('Librerias cargadas exitosamente.')

### 2. Carga y Exploración de Datos Oficiales
El dataset maestro fue procesado mediante el pipeline oficial (`src/data_pipeline.py`) a partir de la fuente de la Red Sismológica Nacional del SGC y el catálogo FDSN. Contiene **32 variables justificadas**, incluyendo métricas territoriales DANE e IGAC.

In [ ]:
# Cargar el dataset maestro limpio y procesado
data_path = '../data/processed/dataset_sismico_santander_medio.csv'
if not os.path.exists(data_path):
    data_path = 'data/processed/dataset_sismico_santander_medio.csv'

df = pd.read_csv(data_path)
print(f'Dimensiones del dataset: {df.shape[0]} observaciones x {df.shape[1]} variables.')
display_cols = ['evento_id', 'fecha_hora_utc', 'magnitud', 'profundidad_km', 'municipio_cercano', 'aceleracion_estimada_pga_g']
df[display_cols].head()

In [ ]:
# Estadisticas descriptivas de las variables sismicas clave
stats_cols = ['magnitud', 'profundidad_km', 'log_energia_joules', 'distancia_nido_los_santos_km', 'aceleracion_estimada_pga_g', 'distancia_municipio_km']
df[stats_cols].describe().T

### 3. Visualización Exploratoria: El Nido de Bucaramanga vs Fallas Corticales
Veamos la distribución de profundidad: se observa claramente la bimodalidad entre la sismicidad superficial (<35 km de fallas activas) y la alta concentración del nido intermedio (~140-165 km).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de profundidad hipocentral
sns.histplot(df['profundidad_km'], bins=35, kde=True, color='#0284c7', ax=ax[0])
ax[0].set_title('Distribución de Profundidad Hipocentral en Santander (km)', fontsize=12, fontweight='bold')
ax[0].set_xlabel('Profundidad (km)')
ax[0].set_ylabel('Frecuencia de Sismos')
ax[0].axvline(150, color='red', linestyle='--', label='Profundidad media Nido Los Santos (~155 km)')
ax[0].legend()

# Scatter Magnitud vs Profundidad
scatter = ax[1].scatter(df['profundidad_km'], df['magnitud'], c=df['aceleracion_estimada_pga_g'], cmap='plasma', alpha=0.7, edgecolors='none')
ax[1].set_title('Magnitud vs Profundidad (Color: Aceleración Pico PGA)', fontsize=12, fontweight='bold')
ax[1].set_xlabel('Profundidad Hipocentral (km)')
ax[1].set_ylabel('Magnitud (Mw)')
cbar = plt.colorbar(scatter, ax=ax[1])
cbar.set_label('PGA Estimada (g)')

plt.tight_layout()
plt.show()

### 4. Preparación de Variables para Modelos No Supervisados
Estandarizamos las 13 variables cuantitativas seleccionadas para eliminar sesgos de escala (por ejemplo, entre Joules de energía y distancias en kilómetros).

In [ ]:
feature_cols = [
    'latitud', 'longitud', 'profundidad_km', 'magnitud', 'log_energia_joules',
    'distancia_nido_los_santos_km', 'distancia_am_bucaramanga_km', 'distancia_municipio_km',
    'aceleracion_estimada_pga_g', 'distancia_hipocentral_km',
    'indice_severidad_hipocentral', 'tasa_sismicidad_local_30d', 'indice_exposicion_humana'
]

X = df[feature_cols].copy().fillna(df[feature_cols].median())
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f'Matriz de caracteristicas escalada: {X_scaled.shape}')

### 5. Comparación Rigurosa de Modelos No Supervisados
Evaluamos 4 arquitecturas distintas:
1. **K-Means Clustering** (Optimizado con curvas de Codo y Silhouette)
2. **DBSCAN** (Agrupamiento basado en densidad espacial y detección de anomalías tectónicas)
3. **Gaussian Mixture Model (GMM)** (Modelado probabilístico con covarianzas elipsoidales)
4. **Clustering Jerárquico Aglomerativo** (Enlace Ward)

Métricas de evaluación:
- **Silhouette Score** (Cohesión vs separación, óptimo cercano a +1)
- **Davies-Bouldin Index** (Similitud inter-cluster, óptimo cercano a 0)
- **Calinski-Harabasz Index** (Razón de dispersión de varianza, mayor es mejor)

In [ ]:
# Metodo del codo y Silhouette para K-Means
k_values = range(2, 8)
inertias, silhouettes = [], []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(k_values, inertias, 'bo-', label='Inercia (Codo)')
ax1.set_xlabel('Número de Clusters (k)')
ax1.set_ylabel('Inercia', color='b')
ax2 = ax1.twinx()
ax2.plot(k_values, silhouettes, 'rs--', label='Silhouette Score')
ax2.set_ylabel('Silhouette Score', color='r')
plt.title('Optimización de K-Means (Inercia vs Silhouette)', fontweight='bold')
plt.show()

In [ ]:
# Entrenamiento de los 4 modelos
km_model = KMeans(n_clusters=4, random_state=42, n_init=15)
km_labels = km_model.fit_predict(X_scaled)

db_model = DBSCAN(eps=1.85, min_samples=8)
db_labels = db_model.fit_predict(X_scaled)

gmm_model = GaussianMixture(n_components=4, covariance_type='full', random_state=42)
gmm_labels = gmm_model.fit_predict(X_scaled)

hier_model = AgglomerativeClustering(n_clusters=4, linkage='ward')
hier_labels = hier_model.fit_predict(X_scaled)

# Funcion evaluadora
def eval_model(name, labels, X_sc):
    mask = labels != -1
    sil = silhouette_score(X_sc[mask], labels[mask]) if len(set(labels[mask])) > 1 else -1
    db = davies_bouldin_score(X_sc[mask], labels[mask]) if len(set(labels[mask])) > 1 else 999
    ch = calinski_harabasz_score(X_sc[mask], labels[mask]) if len(set(labels[mask])) > 1 else 0
    ruido = int(np.sum(labels == -1))
    return {'Modelo': name, 'Clusters': len(set(labels[mask])), 'Silhouette': round(sil, 4), 'Davies-Bouldin': round(db, 4), 'Calinski-Harabasz': round(ch, 1), 'Anomalías Ruido': ruido}

df_comp = pd.DataFrame([
    eval_model('K-Means (k=4)', km_labels, X_scaled),
    eval_model('DBSCAN (eps=1.85, min=8)', db_labels, X_scaled),
    eval_model('Gaussian Mixture (GMM k=4)', gmm_labels, X_scaled),
    eval_model('Jerárquico Ward (k=4)', hier_labels, X_scaled)
])
print('--- COMPARACIÓN CUANTITATIVA DE ENFOQUES NO SUPERVISADOS ---')
df_comp

### 6. Reducción de Dimensionalidad e Importancia de Variables (PCA)
Aplicamos Análisis de Componentes Principales (PCA) para visualizar los clusters en 2D y determinar qué variables tienen mayor poder explicativo.

In [ ]:
pca = PCA(n_components=3, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f'Varianza explicada: PC1={pca.explained_variance_ratio_[0]:.2%}, PC2={pca.explained_variance_ratio_[1]:.2%}, PC3={pca.explained_variance_ratio_[2]:.2%}')
print(f'Varianza acumulada total: {np.sum(pca.explained_variance_ratio_):.2%}')

# Cargas factoriales (Feature Loadings)
loadings = pd.DataFrame(pca.components_.T, index=feature_cols, columns=['PC1 (Amenaza)', 'PC2 (Territorio)', 'PC3 (Enjambre)'])
loadings['Importancia_Media_Abs'] = loadings.abs().mean(axis=1)
loadings.sort_values('Importancia_Media_Abs', ascending=False)

In [ ]:
# Visualizacion en el espacio latente de PCA
plt.figure(figsize=(9, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=km_labels, cmap='tab10', alpha=0.8, edgecolors='k', s=40)
plt.title('Proyección 2D en Componentes Principales (PCA) - Clusters K-Means', fontsize=13, fontweight='bold')
plt.xlabel(f'PC1: Amenaza y Energía ({pca.explained_variance_ratio_[0]:.1%} var)')
plt.ylabel(f'PC2: Proximidad Territorial ({pca.explained_variance_ratio_[1]:.1%} var)')
plt.colorbar(scatter, label='Cluster K-Means')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

### 7. Interpretación Territorial del Riesgo Sísmico (ICRST)
A partir de los clusters no supervisados y las variables de población DANE y geología IGAC, calculamos el **Índice Compuesto de Riesgo Sísmico Territorial (ICRST)**:
$$ICRST = 0.40 \cdot \text{PeligroFísico} + 0.35 \cdot \text{ExposiciónDemográfica} + 0.25 \cdot \text{VulnerabilidadSuelo}$$

Esto resuelve el reto clave: no quedarnos solo en grupos abstractos de ML, sino explicar el riesgo real para la gestión territorial de Santander.

In [ ]:
riesgo_path = '../data/processed/riesgo_territorial_santander.csv'
if not os.path.exists(riesgo_path):
    riesgo_path = 'data/processed/riesgo_territorial_santander.csv'

df_riesgo = pd.read_csv(riesgo_path)
display_riesgo = ['municipio', 'provincia', 'poblacion', 'total_sismos_registrados', 'magnitud_maxima', 'aceleracion_pga_maxima_g', 'icrst_score', 'nivel_riesgo_territorial']
df_riesgo[display_riesgo]

In [ ]:
# Grafico de barras comparativo del Riesgo Territorial
plt.figure(figsize=(12, 6))
colores = ['#dc2626' if 'Muy Alto' in n or 'Alto' in n else '#f59e0b' if 'Medio' in n else '#10b981' for n in df_riesgo['nivel_riesgo_territorial']]
sns.barplot(x='icrst_score', y='municipio', data=df_riesgo, palette=colores)
plt.title('Ranking del Índice Compuesto de Riesgo Sísmico Territorial en Santander (ICRST 0-100)', fontsize=13, fontweight='bold')
plt.xlabel('Puntaje ICRST (0 a 100)')
plt.ylabel('Municipio de Santander')
plt.axvline(50, color='red', linestyle='--', label='Umbral de Riesgo Alto / Monitoreo Continuo')
plt.legend()
plt.tight_layout()
plt.show()

### 8. Conclusiones e Impacto del Proyecto
1. **Disociación Física entre Frecuencia y Daño en Superficie:** Aunque Los Santos y Aratoca concentran más del 70% de los sismos registrados por ubicarse sobre el Nido Sísmico de Bucaramanga, su profundidad promedio (~155 km) amortigua la energía en superficie. Por el contrario, los sismos corticales someros (<35 km) en las proximidades del Área Metropolitana de Bucaramanga y Barrancabermeja generan aceleraciones pico PGA mucho mayores sobre una población que supera 1.3 millones de habitantes.
2. **Superioridad y Complementariedad No Supervisada:** K-Means segmenta nítidamente los 4 regímenes físicos de energía y profundidad, mientras que DBSCAN aísla con precisión 100 eventos anómalos que representan sismos superficiales raros de fallas corticales.
3. **Aplicabilidad en Gestión del Riesgo:** El modelo y su índice ICRST entregan una herramienta reproducible para el Consejo Departamental de Gestión del Riesgo de Desastres (CDGRD Santander) y la Unidad Nacional de Gestión del Riesgo (UNGRD) para priorizar inversiones en reforzamiento estructural y microzonificación sísmica.